In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install transformers sentencepiece

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, T5EncoderModel

In [ ]:
DATA_DIR = '.../codebertmodel'

train_val_df = pd.read_csv(f"{DATA_DIR}/dataset_train_all.csv")
test_df = pd.read_csv(f"{DATA_DIR}/dataset_test_all.csv")

print("Train+val size: ", len(train_val_df))
print("Test size: ", len(test_df))

In [ ]:
def preprocess_df(df):
    df = df.copy()

    # tạo label từ bugs và normals
    def bugs_to_label(bugs_str):
        bugs_str = str(bugs_str).strip()
        return 0 if bugs_str == "[]" else 1

    df["label"] = df["bugs"].apply(bugs_to_label)

    # clean code
    def clean_code(code):
        code = str(code)
        return code.strip()

    df["method"] = df["method"].apply(clean_code)

    # tạo df mới với 2 cột method và label
    df = df[["method", "label"]]

    return df

In [ ]:
train_val_df = preprocess_df(train_val_df)
test_df = preprocess_df(test_df)

In [ ]:
train_val_df.head()
test_df.head()

In [ ]:
print("Train+Val label distribution: ")
print(train_val_df["label"].value_counts())

print("Test label distribution: ")
print(test_df["label"].value_counts())

In [ ]:
# chia train và val theo tỉ lệ 8:2 và giữ tỉ lệ của bugs:normals là 1:7
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.2,
    random_state=42,
    stratify=train_val_df["label"]
)

print("Train size: ", len(train_df))
print("Val size: ", len(test_df))

In [ ]:
print("Train label distribution:")
print(train_df["label"].value_counts(normalize=True))

print("\nValidation label distribution:")
print(val_df["label"].value_counts(normalize=True))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-base")

In [ ]:
def tokenize_dataframe(df, tokenizer, max_len=192):
    encodings = tokenizer(
        df["method"].tolist(),
        truncation=True,
        padding="max_length",
        max_length=max_len
    )
    labels = df["label"].tolist()
    return encodings, labels

In [ ]:
train_encodings, train_labels = tokenize_dataframe(train_df, tokenizer)
val_encodings, val_labels = tokenize_dataframe(val_df, tokenizer)
test_encodings, test_labels = tokenize_dataframe(test_df, tokenizer)

In [ ]:
class CodeT5BugDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx], dtype=torch.long),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx], dtype=torch.long),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }
        return item

In [ ]:
train_dataset = CodeT5BugDataset(train_encodings, train_labels)
val_dataset = CodeT5BugDataset(val_encodings, val_labels)
test_dataset = CodeT5BugDataset(test_encodings, test_labels)

In [ ]:
sample = train_dataset[0]

print("input_ids shape:", sample["input_ids"].shape)
print("attention_mask shape:", sample["attention_mask"].shape)
print("label:", sample["label"])

In [ ]:
BATCH_SIZE = 32

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
# Check 1 batch

batch = next(iter(train_loader))
print("input_ids: ", batch["input_ids"].shape)
print("attention_mask: ", batch["attention_mask"].shape)
print("label: ", batch["label"].shape)

In [ ]:
class CodeT5Classifier(nn.Module):
    def __init__(self, model_name="Salesforce/codet5-base", num_labels=2):
        super().__init__()

        self.encoder = T5EncoderModel.from_pretrained(model_name)

        hidden_size = self.encoder.config.d_model  # 768 for base

        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # outputs.last_hidden_state: (B, L, H)
        hidden_states = outputs.last_hidden_state

        # mean pooling
        pooled = hidden_states.mean(dim=1)  # (B, H)

        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)

        return logits

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CodeT5Classifier().to(device)

In [ ]:
# loss function
class_weights = torch.tensor([1.0, 6.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

In [ ]:
# train loop
from tqdm import tqdm

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0

    for step, batch in enumerate(tqdm(loader, desc="Training", leave=False)):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)

    return avg_loss

In [ ]:
#val loop

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            total_loss += loss.item()

            preds = (torch.softmax(logits, dim=1)[:, 1] >= 0.3).long() #threshold 0.3 -> tăng recall
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, pos_label=1, zero_division=0)
    recall = recall_score(all_labels, all_preds, pos_label=1, zero_division=0)
    f1 = f1_score(all_labels, all_preds, pos_label=1, zero_division=0)

    return avg_loss, acc, precision, recall, f1

In [ ]:
EPOCHS = 5
patience = 2
early_stop_counter = 0

best_val_f1 = 0.0 # tính theo f1 vì dataset lệch

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    val_loss, val_acc, val_prec, val_rec, val_f1 = evaluate(
        model, val_loader, criterion, device
    )

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Acc: {val_acc:.4f} | "
        f"P: {val_prec:.4f} | "
        f"R: {val_rec:.4f} | "
        f"F1: {val_f1:.4f}"
    )

    # SAVE BEST MODEL
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        early_stop_counter = 0

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "val_f1": val_f1
            },
            "best_codet5_model.pt"
        )

        print("Saved best model")
    else:
        early_stop_counter += 1
        print(
            f"⚠️ No improvement ({early_stop_counter}/{patience})"
        )

    # EARLY STOPPING
    if early_stop_counter >= patience:
        print("Early stopping triggered")
        break

In [ ]:
# load best model

checkpoint = torch.load("best_codet5_model.pt", map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

print(
    f"Loaded best model from epoch {checkpoint['epoch']} "
    f"(Val F1 = {checkpoint['val_f1']:.4f})"
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

def test_model(model, loader, device):
    model.eval()

    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            logits = model(input_ids, attention_mask)

            preds = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, pos_label=1, zero_division=0)
    recall = recall_score(all_labels, all_preds, pos_label=1, zero_division=0)
    f1 = f1_score(all_labels, all_preds, pos_label=1, zero_division=0)

    cm = confusion_matrix(all_labels, all_preds)

    return acc, precision, recall, f1, cm

In [ ]:
test_acc, test_prec, test_rec, test_f1, cm = test_model(
    model,
    test_loader,
    device
)

print("\n📊 TEST RESULTS")
print(f"Accuracy : {test_acc:.4f}")
print(f"Precision: {test_prec:.4f}")
print(f"Recall   : {test_rec:.4f}")
print(f"F1-score : {test_f1:.4f}")

print("\nConfusion Matrix:")
print(cm)